# 判決語意向量（Colab GPU 版）

這本筆記本把 Google Drive「判決原始檔」裡的司法院月包，挑出**刑事判決**，用 **BAAI/bge-m3** 在 GPU 上算成語意向量，
存到 Google Drive「判決向量」資料夾。電腦上的 MCP server 直接讀那個資料夾（Google Drive 電腦版會同步），不用手動下載。

**使用方式**
1. 上方選單「執行階段 → 變更執行階段類型」選 **T4 GPU**。
2. 「執行階段 → 全部執行」。第一次會要求授權 Google Drive。
3. **斷線了也沒關係**：重新「全部執行」，已經算好的月份會自動跳過，從下一個月接著算。

約 34 個月、60 萬份判決，T4 大概要 3～5 小時。免費版閒置太久會斷線，分好幾次跑完是正常的；分頁開著比較不容易斷。

每個月算完才會存檔（先寫暫存檔再改名），所以不會留下壞掉的半個檔。

In [ ]:
# ===== 設定（通常不用改）=====
RAW_DIR = "/content/drive/MyDrive/判決原始檔"      # 電腦上是 G:\我的雲端硬碟\判決原始檔
OUT_DIR = "/content/drive/MyDrive/判決向量"        # 電腦上是 G:\我的雲端硬碟\判決向量
MODEL = "BAAI/bge-m3"
MAX_TOKENS = 512         # 每份判決只看事實／理由前段（約 600～800 字）；改 1024 較準但慢一倍
BATCH = 32
INCLUDE_RULINGS = False  # 跟電腦上的資料庫一致：只收刑事「判決」
REPO = "https://github.com/zionchou232-debug/mojLawSplitJSON.git"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
assert os.path.isdir(RAW_DIR), f"找不到 {RAW_DIR}，請確認 Google Drive 裡有「判決原始檔」資料夾"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# 安裝：解 rar 的工具、模型套件；只下載 repo 的 mcp_server 資料夾（不下載法規資料）
!apt-get -qq install -y libarchive-tools unrar > /dev/null
!pip -q install -U sentence-transformers
!rm -rf /content/repo && git clone -q --depth 1 --filter=blob:none --sparse {REPO} /content/repo && cd /content/repo && git sparse-checkout set mcp_server
import sys; sys.path.insert(0, "/content/repo/mcp_server")
from judgments import parse_judgment, fact_passage
from judgment_vectors import save_month
import torch; print("GPU：", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "沒有 GPU！請改成 T4 GPU")

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(MODEL, device="cuda")
model.half()
model.max_seq_length = MAX_TOKENS
print("模型載入完成：", MODEL)

In [ ]:
import json, shutil, subprocess, tempfile, time
from pathlib import Path

def extract(archive: Path, dest: Path):
    for cmd in (["bsdtar", "-xf", str(archive), "-C", str(dest)],
                ["unrar", "x", "-idq", "-o+", str(archive), str(dest) + "/"]):
        if subprocess.run(cmd, capture_output=True).returncode == 0:
            return
    raise RuntimeError(f"解壓失敗：{archive.name}")

def month_passages(archive: Path):
    jids, texts, scanned = [], [], 0
    with tempfile.TemporaryDirectory(dir="/content") as tmp:
        local = Path(tmp) / archive.name
        shutil.copy(archive, local)          # 先複製到 Colab 本機，解壓比較快
        extract(local, Path(tmp))
        local.unlink()
        for p in Path(tmp).rglob("*.json"):
            scanned += 1
            try:
                d = json.loads(p.read_bytes().decode("utf-8-sig"))
            except Exception:
                continue
            j = parse_judgment(d)
            if j is None or (j.kind == "裁定" and not INCLUDE_RULINGS):
                continue
            jids.append(j.jid)
            texts.append(fact_passage(j.full, j.title))
    return jids, texts, scanned

out = Path(OUT_DIR)
archives = sorted(p for p in Path(RAW_DIR).iterdir() if p.suffix.lower() in (".rar", ".zip", ".7z"))
todo = [a for a in archives if not (out / f"{a.stem}.jid.json").exists()]
print(f"共 {len(archives)} 個月包，已完成 {len(archives) - len(todo)} 個，這次要算 {len(todo)} 個")

for i, arc in enumerate(todo, 1):
    t0 = time.time()
    try:
        jids, texts, scanned = month_passages(arc)
    except Exception as e:
        print(f"[{i}/{len(todo)}] {arc.name} 跳過：{e}")
        continue
    if not jids:
        print(f"[{i}/{len(todo)}] {arc.name} 看過 {scanned} 份，沒有刑事判決")
        continue
    vecs = model.encode(texts, batch_size=BATCH, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=False)
    save_month(out, arc.stem, jids, vecs, MODEL)
    print(f"[{i}/{len(todo)}] {arc.name}：看過 {scanned} 份，刑事判決 {len(jids)} 份，"
          f"{time.time() - t0:.0f} 秒")
print("全部完成")

In [ ]:
# 看結果
import numpy as np
files = sorted(Path(OUT_DIR).glob("*.jid.json"))
total = sum(len(json.loads(f.read_text(encoding="utf-8"))["jids"]) for f in files)
size = sum(f.stat().st_size for f in Path(OUT_DIR).glob("*.vec.npy")) / 1e6
print(f"{len(files)} 個月、{total} 份判決的向量，共 {size:.0f} MB，在 {OUT_DIR}")

In [ ]:
# 試試看：用白話描述案情，找事實最像的判決（只看向量，不需要電腦上的資料庫）
QUERY = "朋友借我車，我喝了酒還開車，撞到騎機車的人害他受傷"
q = model.encode([QUERY], normalize_embeddings=True)[0].astype(np.float32)
best = []
for f in files:
    jids = json.loads(f.read_text(encoding="utf-8"))["jids"]
    v = np.load(f.with_name(f.name.replace(".jid.json", ".vec.npy"))).astype(np.float32)
    s = v @ q
    for k in np.argsort(-s)[:5]:
        best.append((float(s[k]), jids[k]))
for sim, jid in sorted(best, reverse=True)[:10]:
    print(f"{sim:.3f}  https://judgment.judicial.gov.tw/FJUD/data.aspx?ty=JD&id={jid}")

## 接到電腦上的 MCP server

1. 在電腦的虛擬環境安裝：`pip install -r mcp_server/requirements-semantic.txt`（fastembed，用來把查詢轉成向量；第一次查詢會下載 bge-m3 約 2.3GB）。
2. 在 Claude Desktop 設定的 `taiwan-law` 加上環境變數：`"env": {"JUDGMENT_VECTOR_DIR": "G:\\我的雲端硬碟\\判決向量"}`，重開 Claude Desktop。
3. `search_judgments` 會自動混合關鍵字與語意搜尋；`find_similar_judgments` 可以找跟某件判決事實相近的案子。

新的月包下載後，再跑一次這本筆記本，只會算新的月份。